In [ ]:
import sys
sys.path.insert(0, '/home/frlab/mj_opt/mj_sim/humanoid')

import numpy as np
import mujoco
import mujoco.viewer
from scipy.spatial import ConvexHull
from dataclasses import dataclass

from core import FloatingBaseRobotState, Pinocchio_Wrapper, Mujoco_Kernel
from utils import SimScheduler, Visualizer, DataLogger, plot_helpers
from control import WholeBodyController, MotionPlanner, TaskSpaceController, GaitScheduler


%load_ext autoreload
%autoreload 2

URDF = '/home/frlab/mj_opt/xmls/systems/g1_description/g1_29dof.urdf'
PKG  = '/home/frlab/mj_opt/xmls/systems/g1_description'
MJCF = '/home/frlab/mj_opt/xmls/systems/g1/scene_29dof.xml'

In [ ]:
PHASE_OFFSET = np.array([0.0, 0.5])   
HEIGHT_SWING = 0.08
LEG_KEYS     = ["L_foot", "R_foot"]
LEG_IDX      = {k: i for i, k in enumerate(LEG_KEYS)}

In [ ]:
class RobotOrchestrator:
    def __init__(self, kernel, wrapper, controller, planner, gait, wbc):
        self.kernel = kernel
        self.wrapper = wrapper
        self.controller = controller
        self.planner = planner
        self.gait_scheduler = gait
        self.wbc = wbc
        
        self.current_ee_pos = np.zeros(3)
        self.current_ee_R = np.eye(3)
        self.current_twist = np.zeros(6)

        self.target_pos = np.zeros(3)
        self.target_R = np.eye(3)
        self.target_twist = np.zeros(6)
        
        self.ctrl_tau = np.zeros(wrapper.nv)

    def on_control(self, t):
        self.kernel.update_robot_state(state)
        self.wrapper.update_model(state.q, state.dq)
        
        phases = np.mod(PHASE_OFFSET + t / self.gait_scheduler.gait_period, 1.0)
        leg_idx = 0
        cycle_phase = phases[leg_idx]
        
        duty = self.gait_scheduler.gait_duty
        if cycle_phase < duty:
            current_phase = (cycle_phase / duty) / (1.0 - duty)
        else:
            current_phase = 0.0
            
        duration = self.gait_scheduler.swing_time
        
        self.target_pos, self.target_R, self.target_twist = self.planner.compute_swing_traj_and_touchdown_humanoid(
            self.wrapper,
            leg=LEG_KEYS[leg_idx],
            phase=current_phase,
            swing_duration=duration,
        )
        
        self.wp_list = self.planner.compute_bezier_trajectory(
            self.target_pos,
            self.target_R,
            self.target_twist,)
        
        self.ctrl_tau = self.wbc.solve(self.wp_list)
        self.kernel.send_control(self.ctrl_tau)
        

    def on_render(self, t, visualizer):
        #visualizer.draw_axes(self.target_pos, self.target_R, size=0.2)

        # ConvexHull (support polygon)
        _, points = kernel.get_foot_contact_state()

        if len(points) >= 3:
            pts = np.array(points)
            try:
                hull = ConvexHull(pts[:, :2])
                z = pts[:, 2].mean()
                verts = pts[hull.vertices]
                for j in range(len(verts)):
                    p1 = np.array([verts[j][0], verts[j][1], z])
                    
                    p2 = np.array([verts[(j+1) % len(verts)][0], verts[(j+1) % len(verts)][1], z])
                    visualizer.draw_line(p1, p2, rgba=(0, 1, 0, 1), thickness=0.005)
            except Exception:
                pass

In [ ]:
PHASE_OFFSET = np.array([0.0, 0.5])   
HEIGHT_SWING = 0.08
LEG_KEYS     = ["L_foot", "R_foot"]
LEG_IDX      = {k: i for i, k in enumerate(LEG_KEYS)}

상태 초기화, wrapper 호출, 커널 호출, 제어기 호출, 초기 ee SE3, SO3, x, y, z 추출 (피노키오 컨벤션)

In [ ]:
state    = FloatingBaseRobotState()
wrapper  = Pinocchio_Wrapper(URDF, PKG)
joint_names = [wrapper.model.names[i] for i in range(2, wrapper.model.njoints)]
kernel   = Mujoco_Kernel(MJCF, joint_names_pin_order=joint_names)
kernel.register_foot_geoms({"L_foot": "left_ankle_roll_link", "R_foot": "right_ankle_roll_link",})
controller = TaskSpaceController(wrapper)   
planner = HumanoidOnlineMotionPlanner(default_step_height=0.5)
gait = GaitScheduler(gait_hz=1.0, duty=0.7)
wbc = WholeBodyController(wrapper)


print("✅ 시스템 호출 완료")

q_init = kernel.q_keyframe
dq_init = np.zeros(wrapper.nv)
wrapper.update_model(q_init, dq_init)

init_oMb = wrapper.oMb
init_oM_Lfoot = wrapper.oM_Lfoot
init_oM_Rfoot = wrapper.oM_Rfoot
init_oM_Lhand = wrapper.oM_Lhand
init_oM_Rhand = wrapper.oM_Rhand

Main Loop (상태 갱신, 제어, 뷰어, 시각화)

In [ ]:
# 실행 직전 초기화 
kernel.reset_to_keyframe()
logger = DataLogger()

# 오케스트레이터 및 스케줄러 실행
orch = RobotOrchestrator(kernel, wrapper, controller, planner, gait, wbc) 
sched = SimScheduler(kernel.model, kernel.data, ctrl_hz=500, render_hz=60)
sched.run(on_control=orch.on_control, on_render=orch.on_render)